<a href="https://colab.research.google.com/github/springboardmentor2468a-lab/Projects_2/blob/Shreya_Merin/week5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import r2_score, mean_squared_error


In [19]:
from google.colab import files
_ = files.upload()



Saving day.csv to day (1).csv
Saving hour.csv to hour (1).csv


In [ ]:
df_day = pd.read_csv("day.csv")
df_hour = pd.read_csv("hour.csv")

print("Day dataset shape:", df_day.shape)
print("Hour dataset shape:", df_hour.shape)


Day dataset shape: (731, 16)
Hour dataset shape: (17379, 17)


In [ ]:
df_day.head(), df_hour.head()


(   instant      dteday  season  yr  mnth  holiday  weekday  workingday  \
 0        1  2011-01-01       1   0     1        0        6           0   
 1        2  2011-01-02       1   0     1        0        0           0   
 2        3  2011-01-03       1   0     1        0        1           1   
 3        4  2011-01-04       1   0     1        0        2           1   
 4        5  2011-01-05       1   0     1        0        3           1   
 
    weathersit      temp     atemp       hum  windspeed  casual  registered  \
 0           2  0.344167  0.363625  0.805833   0.160446     331         654   
 1           2  0.363478  0.353739  0.696087   0.248539     131         670   
 2           1  0.196364  0.189405  0.437273   0.248309     120        1229   
 3           1  0.200000  0.212122  0.590435   0.160296     108        1454   
 4           1  0.226957  0.229270  0.436957   0.186900      82        1518   
 
     cnt  
 0   985  
 1   801  
 2  1349  
 3  1562  
 4  1600  ,
    i

In [ ]:
day_drop_cols = ['instant', 'dteday', 'yr', 'casual', 'registered']
hour_drop_cols = ['instant', 'dteday', 'yr', 'casual', 'registered']

df_day.drop(columns=day_drop_cols, inplace=True, errors='ignore')
df_hour.drop(columns=hour_drop_cols, inplace=True, errors='ignore')


In [ ]:
print("Missing values in Day dataset:\n", df_day.isnull().sum())
print("\nMissing values in Hour dataset:\n", df_hour.isnull().sum())


Missing values in Day dataset:
 season        0
mnth          0
holiday       0
weekday       0
workingday    0
weathersit    0
temp          0
atemp         0
hum           0
windspeed     0
cnt           0
dtype: int64

Missing values in Hour dataset:
 season        0
mnth          0
hr            0
holiday       0
weekday       0
workingday    0
weathersit    0
temp          0
atemp         0
hum           0
windspeed     0
cnt           0
dtype: int64


In [ ]:
def remove_outliers_iqr(df):
    Q1 = df.quantile(0.25)
    Q3 = df.quantile(0.75)
    IQR = Q3 - Q1
    return df[~((df < (Q1 - 1.5 * IQR)) | (df > (Q3 + 1.5 * IQR))).any(axis=1)]

df_day = remove_outliers_iqr(df_day)
df_hour = remove_outliers_iqr(df_hour)

print("After outlier removal:")
print("Day dataset shape:", df_day.shape)
print("Hour dataset shape:", df_hour.shape)


After outlier removal:
Day dataset shape: (693, 11)
Hour dataset shape: (15909, 12)


In [ ]:
X_day = df_day.drop('cnt', axis=1)
y_day = df_day['cnt']

X_hour = df_hour.drop('cnt', axis=1)
y_hour = df_hour['cnt']


In [ ]:
X_day_train, X_day_test, y_day_train, y_day_test = train_test_split(
    X_day, y_day, test_size=0.2, random_state=42
)

X_hour_train, X_hour_test, y_hour_train, y_hour_test = train_test_split(
    X_hour, y_hour, test_size=0.2, random_state=42
)


In [ ]:
# Day dataset
lr_day = LinearRegression()
lr_day.fit(X_day_train, y_day_train)
pred_day_lr = lr_day.predict(X_day_test)

print("Linear Regression - Day Dataset")
print("R2 Score:", r2_score(y_day_test, pred_day_lr))
print("RMSE:", np.sqrt(mean_squared_error(y_day_test, pred_day_lr)))

# Hour dataset
lr_hour = LinearRegression()
lr_hour.fit(X_hour_train, y_hour_train)
pred_hour_lr = lr_hour.predict(X_hour_test)

print("\nLinear Regression - Hour Dataset")
print("R2 Score:", r2_score(y_hour_test, pred_hour_lr))
print("RMSE:", np.sqrt(mean_squared_error(y_hour_test, pred_hour_lr)))


Linear Regression - Day Dataset
R2 Score: 0.5625986792373667
RMSE: 1248.3470464163363

Linear Regression - Hour Dataset
R2 Score: 0.3457911505178609
RMSE: 121.76938557052439


In [ ]:
# Day dataset
dt_day = DecisionTreeRegressor(random_state=42)
dt_day.fit(X_day_train, y_day_train)
pred_day_dt = dt_day.predict(X_day_test)

print("Decision Tree - Day Dataset")
print("R2 Score:", r2_score(y_day_test, pred_day_dt))
print("RMSE:", np.sqrt(mean_squared_error(y_day_test, pred_day_dt)))

# Hour dataset
dt_hour = DecisionTreeRegressor(random_state=42)
dt_hour.fit(X_hour_train, y_hour_train)
pred_hour_dt = dt_hour.predict(X_hour_test)

print("\nDecision Tree - Hour Dataset")
print("R2 Score:", r2_score(y_hour_test, pred_hour_dt))
print("RMSE:", np.sqrt(mean_squared_error(y_hour_test, pred_hour_dt)))


Decision Tree - Day Dataset
R2 Score: 0.4347478909242487
RMSE: 1419.1110729643653

Decision Tree - Hour Dataset
R2 Score: 0.7386284584713759
RMSE: 76.96779714667404


In [ ]:
rf_hour = RandomForestRegressor(random_state=42)
rf_hour.fit(X_hour_train, y_hour_train)

rf_preds = rf_hour.predict(X_hour_test)

print("Random Forest - Hour Dataset")
print("R2 Score:", r2_score(y_hour_test, rf_preds))
print("RMSE:", np.sqrt(mean_squared_error(y_hour_test, rf_preds)))


Random Forest - Hour Dataset
R2 Score: 0.8609514334262163
RMSE: 56.13880521954183


In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

rf = RandomForestRegressor(random_state=42)

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1
)

grid_search.fit(X_hour_train, y_hour_train)

print("Best Parameters:", grid_search.best_params_)


Best Parameters: {'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}


In [ ]:
best_rf = grid_search.best_estimator_
best_preds = best_rf.predict(X_hour_test)

print("Tuned Random Forest - Hour Dataset")
print("R2 Score:", r2_score(y_hour_test, best_preds))
print("RMSE:", np.sqrt(mean_squared_error(y_hour_test, best_preds)))


Tuned Random Forest - Hour Dataset
R2 Score: 0.8617047432341265
RMSE: 55.98652984274267


In [ ]:
feature_importance = pd.Series(
    best_rf.feature_importances_,
    index=X_hour.columns
).sort_values(ascending=False)

feature_importance.head(10)


,0
hr,0.633989
temp,0.124451
hum,0.052928
workingday,0.041908
atemp,0.034389
windspeed,0.024860
season,0.022821
weekday,0.022233
weathersit,0.022206
mnth,0.020214
